# 📄 Dispatcher Service — Vertex AI Workbench Notebook

Run this notebook inside **Vertex AI Workbench** — credentials are automatic via the managed service account.

**Pipeline Steps:**
1. Install extra dependencies (only what's not pre-installed)
2. Authenticate & configure
3. Classify a PDF from GCS using Gemini Flash
4. Simulate Pub/Sub routing
5. Test the live Cloud Run dispatcher endpoint

## Step 1 — Install Extra Dependencies
*(google-cloud-* and vertexai are pre-installed in Workbench)*

In [ ]:
!pip install pypdf --quiet
print('Done ✅')

## Step 2 — Configuration
No API key needed — Workbench authenticates automatically.

In [ ]:
import json, base64, io
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import pubsub_v1, storage
from pypdf import PdfReader

# ── Project settings ─────────────────────────────────────────────────────────
PROJECT_ID = "ml-deployment-482112-490513"
REGION     = "us-central1"

# ── GCS file to classify ──────────────────────────────────────────────────────
GCS_BUCKET   = "raw-documents-ml-deployment-482112-490513-dev"
GCS_FILENAME = "test-invoice.pdf"
GCS_URI      = f"gs://{GCS_BUCKET}/{GCS_FILENAME}"

# ── Pub/Sub topics ────────────────────────────────────────────────────────────
INVOICE_TOPIC     = "process-invoice"
CONTRACT_TOPIC    = "process-contract"
DEAD_LETTER_TOPIC = "dead-letter-topic"

# ── Dispatcher Cloud Run URL ──────────────────────────────────────────────────
DISPATCHER_URL = "https://dispatcher-699339551124.us-central1.run.app/"

# Initialise Vertex AI SDK with project credentials (automatic in Workbench)
vertexai.init(project=PROJECT_ID, location=REGION)

storage_client = storage.Client(project=PROJECT_ID)
publisher      = pubsub_v1.PublisherClient()

print(f"Project : {PROJECT_ID}")
print(f"File    : {GCS_URI}")
print("Ready ✅")

## Step 3a — Classify using Gemini (Native GCS URI)
Vertex AI reads the PDF directly from GCS — no download needed.

In [ ]:
CLASSIFICATION_PROMPT = """You are a document classifier. Analyze the document and identify its type.
Respond with ONLY one of these exact words: INVOICE, CONTRACT, TAX_FORM, UNKNOWN.
No explanation. No punctuation. Just the single word."""

model    = GenerativeModel("gemini-1.5-flash-001")
document = Part.from_uri(mime_type="application/pdf", uri=GCS_URI)

response = model.generate_content([CLASSIFICATION_PROMPT, document])
doc_type = response.text.strip().upper()

print(f"🤖 Gemini classified: {doc_type}")

## Step 3b — (Alternative) Classify using Extracted Text
Use if Step 3a returns a 404 for the model version.

In [ ]:
# Download PDF and extract text
blob      = storage_client.bucket(GCS_BUCKET).blob(GCS_FILENAME)
pdf_bytes = blob.download_as_bytes()
reader    = PdfReader(io.BytesIO(pdf_bytes))
doc_text  = "\n".join(page.extract_text() or "" for page in reader.pages)[:2000]

print(f"Extracted {len(doc_text)} chars from {GCS_URI}")
print("─" * 60)
print(doc_text[:300], "...")

# Classify using extracted text
model    = GenerativeModel("gemini-1.5-flash-001")
response = model.generate_content(f"{CLASSIFICATION_PROMPT}\n\nDocument text:\n\n{doc_text}")
doc_type = response.text.strip().upper()

print(f"\n🤖 Gemini classified: {doc_type}")

## Step 4 — Simulate Pub/Sub Routing
Set `DRY_RUN = False` to actually publish the message.

In [ ]:
DRY_RUN = True   # Change to False to publish to Pub/Sub

if "INVOICE" in doc_type:
    target_topic = INVOICE_TOPIC
elif "CONTRACT" in doc_type:
    target_topic = CONTRACT_TOPIC
else:
    target_topic = DEAD_LETTER_TOPIC

payload = {
    "gcs_uri":       GCS_URI,
    "bucket":        GCS_BUCKET,
    "filename":      GCS_FILENAME,
    "document_type": doc_type,
}

print(f"📨 Document type : {doc_type}")
print(f"📬 Target topic  : {target_topic}")
print(f"📦 Payload       :\n{json.dumps(payload, indent=2)}")

if not DRY_RUN:
    topic_path = publisher.topic_path(PROJECT_ID, target_topic)
    future     = publisher.publish(topic_path, json.dumps(payload).encode("utf-8"))
    print(f"\n✅ Published! Message ID: {future.result()}")
else:
    print("\n⚠️  DRY_RUN mode — not published. Set DRY_RUN=False to publish.")

## Step 5 — Test the Live Cloud Run Dispatcher Endpoint
Sends a simulated Pub/Sub push event to your deployed dispatcher service.

In [ ]:
import requests

# Build the simulated GCS event (same format Pub/Sub sends)
gcs_event       = json.dumps({"bucket": GCS_BUCKET, "name": GCS_FILENAME})
encoded_event   = base64.b64encode(gcs_event.encode()).decode()
pubsub_envelope = {"message": {"data": encoded_event}}

print(f"Sending POST request to {DISPATCHER_URL}")
resp = requests.post(DISPATCHER_URL, json=pubsub_envelope, timeout=60)

print(f"Status   : {resp.status_code}")
print(f"Response : {resp.json()}")